# 01 — EDA: Apontamentos de Equipamentos

**Objetivo:** Entender a estrutura dos apontamentos de estado dos equipamentos — distribuição temporal, frotas, classes operacionais e padrões relevantes para o modelo preditivo.

**Dataset:** `desenvolver_apontamentos.parquet` — 377.907 registros | Jan–Jun 2025 | 47 equipamentos


In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from src.ingestion import load_apontamentos

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(40)


polars.config.Config

## 1. Carregamento e Visão Geral

In [2]:
df = load_apontamentos().collect()

print(f"Registros: {df.shape[0]:,}")
print(f"Colunas  : {df.shape[1]}")
print(f"Período  : {df['Inicio'].min()} → {df['Fim'].max()}")
print(f"\nSchema:")
print(df.schema)


Registros: 377,907
Colunas  : 7
Período  : 2025-01-01 00:00:00 → 2025-07-01 00:00:00

Schema:
Schema({'Id': Int64, 'Inicio': Datetime(time_unit='ns', time_zone=None), 'Fim': Datetime(time_unit='ns', time_zone=None), 'Tag': String, 'Frota': String, 'Tipo': String, 'Classe': String})


In [3]:
# Qualidade dos dados — nulos
print("Nulos por coluna:")
print(df.null_count())


Nulos por coluna:
shape: (1, 7)
┌─────┬────────┬─────┬─────┬───────┬──────┬────────┐
│ Id  ┆ Inicio ┆ Fim ┆ Tag ┆ Frota ┆ Tipo ┆ Classe │
│ --- ┆ ---    ┆ --- ┆ --- ┆ ---   ┆ ---  ┆ ---    │
│ u32 ┆ u32    ┆ u32 ┆ u32 ┆ u32   ┆ u32  ┆ u32    │
╞═════╪════════╪═════╪═════╪═══════╪══════╪════════╡
│ 0   ┆ 0      ┆ 0   ┆ 0   ┆ 0     ┆ 0    ┆ 0      │
└─────┴────────┴─────┴─────┴───────┴──────┴────────┘


In [4]:
df.head(5)


Id,Inicio,Fim,Tag,Frota,Tipo,Classe
i64,datetime[ns],datetime[ns],str,str,str,str
23462554,2025-01-04 19:12:54,2025-01-04 19:40:29,"""CA65789""","""793-D 2S""","""Caminhao""","""Operando"""
23462555,2025-01-04 19:12:28,2025-01-04 19:38:15,"""CA65908""","""793-D 3S""","""Caminhao""","""Operando"""
23462556,2025-01-04 19:14:07,2025-01-04 19:18:29,"""CA65915""","""793-D 4S""","""Caminhao""","""Parado"""
23462759,2025-01-04 19:20:36,2025-01-04 19:27:44,"""CA5926""","""793-D 5S""","""Caminhao""","""Parado"""
23462762,2025-01-04 19:18:29,2025-01-04 19:23:51,"""CA65915""","""793-D 4S""","""Caminhao""","""Operando"""


## 2. Distribuição de Frotas e Equipamentos

In [5]:
frota_counts = (
    df.group_by(["Tipo", "Frota"])
      .agg(
          pl.len().alias("apontamentos"),
          pl.col("Tag").n_unique().alias("n_equipamentos"),
      )
      .sort("apontamentos", descending=True)
)
print(frota_counts)


shape: (5, 4)
┌─────────────┬───────────────────┬──────────────┬────────────────┐
│ Tipo        ┆ Frota             ┆ apontamentos ┆ n_equipamentos │
│ ---         ┆ ---               ┆ ---          ┆ ---            │
│ str         ┆ str               ┆ u32          ┆ u32            │
╞═════════════╪═══════════════════╪══════════════╪════════════════╡
│ Caminhao    ┆ 793-D 5S          ┆ 146331       ┆ 14             │
│ Caminhao    ┆ 793-D 4S          ┆ 93820        ┆ 12             │
│ Escavadeira ┆ LeTourneau L 1850 ┆ 63899        ┆ 10             │
│ Caminhao    ┆ 793-D 2S          ┆ 42867        ┆ 5              │
│ Caminhao    ┆ 793-D 3S          ┆ 30990        ┆ 6              │
└─────────────┴───────────────────┴──────────────┴────────────────┘


In [6]:
fig = px.bar(
    frota_counts.to_pandas(),
    x="Frota", y="apontamentos", color="Tipo",
    text="n_equipamentos",
    title="Apontamentos por Frota (número sobre a barra = equipamentos únicos)",
    labels={"apontamentos": "Nº de Apontamentos", "Frota": "Modelo de Frota"},
    color_discrete_map={"Caminhao": "#1f77b4", "Escavadeira": "#ff7f0e"},
)
fig.update_traces(texttemplate="%{text} equip.", textposition="outside")
fig.update_layout(height=450)
fig.show()


## 3. Classes Operacionais (Estado dos Equipamentos)

In [7]:
classe_tipo = (
    df.group_by(["Tipo", "Classe"])
      .len()
      .sort(["Tipo", "len"], descending=[False, True])
)

fig = px.bar(
    classe_tipo.to_pandas(),
    x="Classe", y="len", color="Tipo", barmode="group",
    title="Distribuição de Classes Operacionais por Tipo de Equipamento",
    labels={"len": "Nº de Apontamentos", "Classe": "Classe Operacional"},
    color_discrete_map={"Caminhao": "#1f77b4", "Escavadeira": "#ff7f0e"},
    category_orders={"Classe": ["Operando", "Parado", "Hibernando", "Manutenção"]},
)
fig.update_layout(height=420)
fig.show()


In [8]:
# Proporção de cada classe
classe_pct = (
    df.group_by("Classe")
      .len()
      .with_columns((pl.col("len") / pl.col("len").sum() * 100).round(1).alias("pct"))
      .sort("len", descending=True)
)
print(classe_pct)


shape: (4, 3)
┌────────────┬────────┬──────┐
│ Classe     ┆ len    ┆ pct  │
│ ---        ┆ ---    ┆ ---  │
│ str        ┆ u32    ┆ f64  │
╞════════════╪════════╪══════╡
│ Operando   ┆ 182527 ┆ 48.3 │
│ Parado     ┆ 106531 ┆ 28.2 │
│ Hibernando ┆ 54868  ┆ 14.5 │
│ Manutenção ┆ 33981  ┆ 9.0  │
└────────────┴────────┴──────┘


In [9]:
fig = px.pie(
    classe_pct.to_pandas(),
    names="Classe", values="len",
    title="Proporção Global das Classes Operacionais",
    hole=0.4,
)
fig.update_traces(textinfo="label+percent")
fig.show()


## 4. Duração dos Apontamentos

In [10]:
df_dur = df.with_columns(
    ((pl.col("Fim") - pl.col("Inicio")).dt.total_minutes()).alias("duracao_min")
)

print("Estatísticas de duração (minutos):")
print(df_dur["duracao_min"].describe())

# Anomalias de duração
print(f"\nApontamentos com 0 min  : {(df_dur['duracao_min'] == 0).sum():,} ({(df_dur['duracao_min'] == 0).mean()*100:.1f}%)")
print(f"Apontamentos com 60 min : {(df_dur['duracao_min'] == 60).sum():,} ({(df_dur['duracao_min'] == 60).mean()*100:.1f}%)")
print(f"Apontamentos com > 60min: {(df_dur['duracao_min'] > 60).sum():,}")


Estatísticas de duração (minutos):


shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 377907.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 29.351983 │
│ std        ┆ 24.22121  │
│ min        ┆ 0.0       │
│ 25%        ┆ 6.0       │
│ 50%        ┆ 22.0      │
│ 75%        ┆ 60.0      │
│ max        ┆ 60.0      │
└────────────┴───────────┘

Apontamentos com 0 min  : 25,162 (6.7%)
Apontamentos com 60 min : 114,924 (30.4%)
Apontamentos com > 60min: 0


In [11]:
# Histograma de duração (excluindo 0 e limitando a 60 para visualização clara)
df_dur_plot = df_dur.filter(
    (pl.col("duracao_min") > 0) & (pl.col("duracao_min") <= 60)
)

fig = px.histogram(
    df_dur_plot.to_pandas(),
    x="duracao_min", color="Classe", nbins=60,
    title="Distribuição de Duração dos Apontamentos (excl. 0 min, até 60 min)",
    labels={"duracao_min": "Duração (minutos)", "count": "Frequência"},
    category_orders={"Classe": ["Operando", "Parado", "Hibernando", "Manutenção"]},
    barmode="overlay", opacity=0.7,
)
fig.update_layout(height=420)
fig.show()


In [12]:
# Nota: pico em 60 min é truncamento de janela de apontamento
# Os apontamentos são registrados em janelas de até 60 minutos
# Equipamentos que ficam mais de 60 min em um estado geram múltiplos registros consecutivos
duracao_por_classe = (
    df_dur.group_by("Classe")
      .agg(
          pl.col("duracao_min").mean().round(1).alias("media_min"),
          pl.col("duracao_min").median().alias("mediana_min"),
          pl.col("duracao_min").std().round(1).alias("std_min"),
      )
      .sort("media_min", descending=True)
)
print("Duração média por Classe (min):")
print(duracao_por_classe)


Duração média por Classe (min):
shape: (4, 4)
┌────────────┬───────────┬─────────────┬─────────┐
│ Classe     ┆ media_min ┆ mediana_min ┆ std_min │
│ ---        ┆ ---       ┆ ---         ┆ ---     │
│ str        ┆ f64       ┆ f64         ┆ f64     │
╞════════════╪═══════════╪═════════════╪═════════╡
│ Hibernando ┆ 60.0      ┆ 60.0        ┆ 0.5     │
│ Manutenção ┆ 51.6      ┆ 60.0        ┆ 16.9    │
│ Operando   ┆ 26.0      ┆ 21.0        ┆ 21.4    │
│ Parado     ┆ 12.3      ┆ 7.0         ┆ 15.9    │
└────────────┴───────────┴─────────────┴─────────┘


## 5. Evolução Temporal

In [13]:
df_mensal = (
    df.with_columns(pl.col("Inicio").dt.month().alias("mes"))
      .group_by(["mes", "Classe"])
      .len()
      .sort(["mes", "Classe"])
)

fig = px.line(
    df_mensal.to_pandas(),
    x="mes", y="len", color="Classe", markers=True,
    title="Volume de Apontamentos por Mês e Classe",
    labels={"len": "Nº de Apontamentos", "mes": "Mês (2025)"},
    category_orders={"Classe": ["Operando", "Parado", "Hibernando", "Manutenção"]},
)
fig.update_xaxes(tickvals=list(range(1, 7)), ticktext=["Jan","Fev","Mar","Abr","Mai","Jun"])
fig.update_layout(height=420)
fig.show()


In [14]:
# Taxa de disponibilidade mensal (Operando / total) por tipo
disponibilidade = (
    df.with_columns(pl.col("Inicio").dt.month().alias("mes"))
      .group_by(["mes", "Tipo"])
      .agg(
          pl.len().alias("total"),
          (pl.col("Classe") == "Operando").sum().alias("operando"),
      )
      .with_columns((pl.col("operando") / pl.col("total") * 100).round(1).alias("disponibilidade_pct"))
      .sort(["Tipo", "mes"])
)

fig = px.line(
    disponibilidade.to_pandas(),
    x="mes", y="disponibilidade_pct", color="Tipo", markers=True,
    title="Taxa de Disponibilidade (% Operando) por Tipo de Equipamento",
    labels={"disponibilidade_pct": "% Operando", "mes": "Mês (2025)"},
    color_discrete_map={"Caminhao": "#1f77b4", "Escavadeira": "#ff7f0e"},
)
fig.update_xaxes(tickvals=list(range(1, 7)), ticktext=["Jan","Fev","Mar","Abr","Mai","Jun"])
fig.update_layout(height=400, yaxis_range=[0, 60])
fig.show()


## 6. Perfil Individual dos Equipamentos

In [15]:
# Heatmap: equipamento x classe (proporção do tempo)
equip_classe = (
    df.group_by(["Tag", "Frota", "Classe"])
      .len()
      .with_columns(
          (pl.col("len") / pl.col("len").sum().over("Tag") * 100)
            .round(1).alias("pct")
      )
)

pivot = equip_classe.pivot(
    on="Classe", index=["Tag", "Frota"], values="pct"
).fill_null(0).sort("Frota")

fig = px.imshow(
    pivot.select(["Operando", "Parado", "Hibernando", "Manutenção"]).to_pandas(),
    y=pivot["Tag"].to_list(),
    title="Heatmap: % de Apontamentos por Equipamento e Classe",
    labels={"x": "Classe", "y": "Equipamento (TAG)", "color": "% apontamentos"},
    color_continuous_scale="RdYlGn",
    aspect="auto",
)
fig.update_layout(height=700)
fig.show()


## 7. Padrões de Manutenção por Frota

In [16]:
# Frequência de entrada em manutenção por equipamento e mês
manutencao = (
    df.filter(pl.col("Classe") == "Manutenção")
      .with_columns(pl.col("Inicio").dt.month().alias("mes"))
      .group_by(["Frota", "mes"])
      .len()
      .sort(["Frota", "mes"])
)

fig = px.bar(
    manutencao.to_pandas(),
    x="mes", y="len", color="Frota", barmode="group",
    title="Eventos de Manutenção por Frota e Mês",
    labels={"len": "Nº de Eventos de Manutenção", "mes": "Mês (2025)"},
)
fig.update_xaxes(tickvals=list(range(1, 7)), ticktext=["Jan","Fev","Mar","Abr","Mai","Jun"])
fig.update_layout(height=420)
fig.show()


## 8. Insights e Conclusões

### Achados Principais

| # | Insight | Relevância para o Modelo |
|---|---------|--------------------------|
| I1 | **Dados sem nulos** — qualidade excelente, pronto para uso | Baixo risco de bias por imputação |
| I2 | **47 equipamentos únicos**: 37 caminhões (5 frotas 793-D) + 10 escavadeiras (LeTourneau L 1850) | Segmentar modelo por tipo |
| I3 | **Operando = 48%** dos apontamentos, mas **Parado = 28%** — alta ociosidade | Parado ≠ Manutenção: parada pode preceder Don't Go |
| I4 | **30% dos apontamentos têm exatamente 60 min** — janela de registro truncada em 60 min | Ao fazer join com telemetria, considerar sequências contínuas |
| I5 | **25.162 apontamentos com 0 min** (6,7%) — transições instantâneas ou erros de registro | Filtrar ou tratar no pipeline Bronze→Silver |
| I6 | **Disponibilidade de Escavadeiras é ~30%** vs ~50% dos Caminhões — perfis de uso muito diferentes | Modelos separados por tipo podem ser necessários |
| I7 | **793-D 5S é a maior frota** (14 equipamentos, 39% dos apontamentos) — mais dados para treino | Frota 5S terá maior representatividade no modelo |

### Próximos Passos
- Notebook `02_EDA_telemetria.ipynb`: explorar alarmes, padrões Don't Go e hipóteses H1–H7
- Pipeline `transformation.py`: join apontamentos ↔ telemetria via TAG + janela temporal
- Tratar apontamentos com 0 min na camada Bronze
